# Bab 14: Inferensi Variasional

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab14/data_berita.py": "\"\"\"Korpus berita daerah sintetis yang dibangkitkan dari model LDA (ilustrasi).\n\nEnam topik dengan kosakata khasnya, ditambah kata umum yang muncul di semua\ntopik. Setiap dokumen memiliki campuran topiknya sendiri.\n\"\"\"\nimport numpy as np\n\nTOPIK = {\n    \"sawit\":    \"sawit kebun petani tbs harga pabrik plasma replanting cpo lahan \"\n                \"produksi koperasi bibit pupuk ekspor\",\n    \"banjir\":   \"banjir hujan sungai warga pengungsi bpbd tanggul evakuasi debit \"\n                \"longsor genangan bantuan perahu rendam siaga\",\n    \"pilkada\":  \"pilkada calon kpu suara pemilih kampanye debat bawaslu partai \"\n                \"koalisi survei tps paslon dukungan pencoblosan\",\n    \"wisata\":   \"wisata pantai wisatawan hotel festival budaya kuliner danau homestay \"\n                \"tiket libur pengunjung desa pesona promosi\",\n    \"bola\":     \"pemain gol klub pelatih liga laga stadion suporter skor kompetisi \"\n                \"babak tim juara kiper menang\",\n    \"dbd\":      \"dbd nyamuk fogging puskesmas pasien demam kasus jentik dinkes rsud \"\n                \"trombosit abate pencegahan kesehatan\",\n}\nUMUM = \"kata pemerintah daerah kabupaten provinsi tahun hari masyarakat kota bupati\".split()\n\ndef buat_korpus(n_dok=1200, panjang=80, alpha=0.15, seed=0):\n    rng = np.random.default_rng(seed)\n    kosakata = sorted({w for v in TOPIK.values() for w in v.split()} | set(UMUM))\n    iw = {w: i for i, w in enumerate(kosakata)}\n    phi = np.full((len(TOPIK), len(kosakata)), 1e-3)     # distribusi kata per topik\n    for k, v in enumerate(TOPIK.values()):\n        kata = v.split()\n        phi[k, [iw[w] for w in kata]] += rng.dirichlet(np.ones(len(kata)) * 2) * 0.75\n        phi[k, [iw[w] for w in UMUM]] += 0.25 / len(UMUM)\n    phi /= phi.sum(1, keepdims=True)\n    theta = rng.dirichlet(np.ones(len(TOPIK)) * alpha, n_dok)  # campuran per dokumen\n    dok = []\n    for d in range(n_dok):\n        z = rng.choice(len(TOPIK), panjang, p=theta[d])\n        dok.append(\" \".join(kosakata[rng.choice(len(kosakata), p=phi[k])] for k in z))\n    return dok, theta, list(TOPIK)\n",
"bab14/lda_berita.py": "import numpy as np\nfrom sklearn.feature_extraction.text import CountVectorizer\nfrom sklearn.decomposition import LatentDirichletAllocation\nfrom data_berita import buat_korpus, TOPIK\n\ndok, theta, nama = buat_korpus()\nlatih, uji = dok[:1000], dok[1000:]\ncv = CountVectorizer(token_pattern=r\"[a-z]+\")\nXl, Xu = cv.fit_transform(latih), cv.transform(uji)       # bag of words  #<1>\nkata = cv.get_feature_names_out()\n\nlda = LatentDirichletAllocation(6, doc_topic_prior=0.1, topic_word_prior=0.01,\n                                learning_method=\"batch\", random_state=0).fit(Xl)  #<2>\nlabel = []\nfor k, b in enumerate(lda.components_):                   # kata teratas  #<3>\n    atas = kata[np.argsort(b)[::-1][:6]]\n    label.append(max(TOPIK, key=lambda t: len(set(atas) & set(TOPIK[t].split()))))\n    print(f\"topik {k} ({label[-1]:7s}): \" + \" \".join(atas))\nd = 1000 + np.argmax(theta[1000:].max(1) < 0.7)         # dokumen uji campuran  #<4>\nth = lda.transform(Xu[d - 1000])[0]\nest = {label[k]: p for k, p in enumerate(th)}\nprint(f\"dokumen #{d}  \" + \" \".join(f\"{t:>7s}\" for t in nama))\nprint(\"  taksiran    \" + \" \".join(f\"{est.get(t, 0):7.2f}\" for t in nama))\nprint(\"  sebenarnya  \" + \" \".join(f\"{p:7.2f}\" for p in theta[d]))\nfor K in [3, 6, 12]:                                      # perplexity data uji\n    m = LatentDirichletAllocation(K, random_state=0).fit(Xl)\n    print(f\"K = {K:2d}: perplexity uji {m.perplexity(Xu):.1f}\")\n",
"bab14/vb_campuran.py": "import numpy as np\nfrom sklearn.mixture import GaussianMixture, BayesianGaussianMixture\n\n# Empat kelompok titik 2D (sintetis, ilustrasi), 400 titik.\nrng = np.random.default_rng(1)\npusat = np.array([[0, 0], [4, 1], [1, 4], [5, 5]])\nX = np.vstack([rng.multivariate_normal(m, [[0.6, 0.2], [0.2, 0.5]], 100) for m in pusat])\n\nK = 10                                                   # sengaja terlalu banyak\nem = GaussianMixture(K, n_init=3, random_state=0).fit(X)  #<1>\nvb = BayesianGaussianMixture(n_components=K, weight_concentration_prior=1e-3,\n                             weight_concentration_prior_type=\"dirichlet_distribution\",\n                             n_init=3, max_iter=1000, random_state=0).fit(X)  #<2>\nprint(\"bobot EM :\", \" \".join(f\"{w:.2f}\" for w in np.sort(em.weights_)[::-1]))\nprint(\"bobot VB :\", \" \".join(f\"{w:.2f}\" for w in np.sort(vb.weights_)[::-1]))\nprint(\"komponen aktif (bobot > 1%): EM\", np.sum(em.weights_ > 0.01),\n      \"| VB\", np.sum(vb.weights_ > 0.01))\nprint(f\"ELBO akhir VB (seluruh data): {vb.lower_bound_:.1f}\")  #<3>\nXu = np.vstack([rng.multivariate_normal(m, [[0.6, 0.2], [0.2, 0.5]], 100) for m in pusat])\nprint(f\"log-lik. uji per titik: EM {em.score(Xu):.3f}, VB {vb.score(Xu):.3f}\")\n",
"bab14/vb_gauss.py": "import numpy as np\n\n# Hasil gabah kering (ton/ha) dari 10 petak uji (sintetis, ilustrasi).\nx = np.array([5.8, 6.4, 5.1, 6.9, 5.5, 6.2, 7.1, 5.9, 6.6, 6.0])\nN, xb = len(x), x.mean()\nmu0, lam0, a0, b0 = 5.0, 1.0, 1.0, 1.0         # prior Normal-Gamma  #<1>\n\nE_tau = 1.0                                     # tebakan awal E[tau]\nfor it in range(5):                             # pembaruan bergantian  #<2>\n    muN = (lam0 * mu0 + N * xb) / (lam0 + N)    # q(mu) = N(muN, 1/lamN)\n    lamN = (lam0 + N) * E_tau\n    E_mu, E_mu2 = muN, muN ** 2 + 1 / lamN\n    aN = a0 + (N + 1) / 2                       # q(tau) = Gam(aN, bN)\n    bN = b0 + 0.5 * (np.sum(x ** 2) - 2 * E_mu * x.sum() + N * E_mu2\n                     + lam0 * (E_mu2 - 2 * mu0 * E_mu + mu0 ** 2))\n    E_tau = aN / bN\n    print(f\"iterasi {it}: E[mu] = {muN:.4f}, sd[mu] = {lamN ** -0.5:.4f},\"\n          f\" E[tau] = {E_tau:.4f}\")\n\n# posterior eksak Normal-Gamma (Bab 3) untuk pembanding  #<3>\nlamE, muE = lam0 + N, (lam0 * mu0 + N * xb) / (lam0 + N)\naE = a0 + N / 2\nbE = b0 + 0.5 * (np.sum((x - xb) ** 2) + lam0 * N * (xb - mu0) ** 2 / (lam0 + N))\nsd_mu_eksak = np.sqrt(bE / (aE - 1) / lamE)     # marginal mu: Student-t\nprint(f\"eksak     : E[mu] = {muE:.4f}, sd[mu] = {sd_mu_eksak:.4f}, E[tau] = {aE / bE:.4f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab14')
print('folder kerja:', os.getcwd())

## Listing 14.1: Mean-field untuk rerata dan presisi Gaussian

Berkas: `kode/bab14/vb_gauss.py`

In [ ]:
import numpy as np

# Hasil gabah kering (ton/ha) dari 10 petak uji (sintetis, ilustrasi).
x = np.array([5.8, 6.4, 5.1, 6.9, 5.5, 6.2, 7.1, 5.9, 6.6, 6.0])
N, xb = len(x), x.mean()
mu0, lam0, a0, b0 = 5.0, 1.0, 1.0, 1.0         # prior Normal-Gamma  # (1)

E_tau = 1.0                                     # tebakan awal E[tau]
for it in range(5):                             # pembaruan bergantian  # (2)
    muN = (lam0 * mu0 + N * xb) / (lam0 + N)    # q(mu) = N(muN, 1/lamN)
    lamN = (lam0 + N) * E_tau
    E_mu, E_mu2 = muN, muN ** 2 + 1 / lamN
    aN = a0 + (N + 1) / 2                       # q(tau) = Gam(aN, bN)
    bN = b0 + 0.5 * (np.sum(x ** 2) - 2 * E_mu * x.sum() + N * E_mu2
                     + lam0 * (E_mu2 - 2 * mu0 * E_mu + mu0 ** 2))
    E_tau = aN / bN
    print(f"iterasi {it}: E[mu] = {muN:.4f}, sd[mu] = {lamN ** -0.5:.4f},"
          f" E[tau] = {E_tau:.4f}")

# posterior eksak Normal-Gamma (Bab 3) untuk pembanding  # (3)
lamE, muE = lam0 + N, (lam0 * mu0 + N * xb) / (lam0 + N)
aE = a0 + N / 2
bE = b0 + 0.5 * (np.sum((x - xb) ** 2) + lam0 * N * (xb - mu0) ** 2 / (lam0 + N))
sd_mu_eksak = np.sqrt(bE / (aE - 1) / lamE)     # marginal mu: Student-t
print(f"eksak     : E[mu] = {muE:.4f}, sd[mu] = {sd_mu_eksak:.4f}, E[tau] = {aE / bE:.4f}")

## Listing 14.2: EM dan inferensi variasional dengan sepuluh komponen untuk data empat kelompok

Berkas: `kode/bab14/vb_campuran.py`

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture

# Empat kelompok titik 2D (sintetis, ilustrasi), 400 titik.
rng = np.random.default_rng(1)
pusat = np.array([[0, 0], [4, 1], [1, 4], [5, 5]])
X = np.vstack([rng.multivariate_normal(m, [[0.6, 0.2], [0.2, 0.5]], 100) for m in pusat])

K = 10                                                   # sengaja terlalu banyak
em = GaussianMixture(K, n_init=3, random_state=0).fit(X)  # (1)
vb = BayesianGaussianMixture(n_components=K, weight_concentration_prior=1e-3,
                             weight_concentration_prior_type="dirichlet_distribution",
                             n_init=3, max_iter=1000, random_state=0).fit(X)  # (2)
print("bobot EM :", " ".join(f"{w:.2f}" for w in np.sort(em.weights_)[::-1]))
print("bobot VB :", " ".join(f"{w:.2f}" for w in np.sort(vb.weights_)[::-1]))
print("komponen aktif (bobot > 1%): EM", np.sum(em.weights_ > 0.01),
      "| VB", np.sum(vb.weights_ > 0.01))
print(f"ELBO akhir VB (seluruh data): {vb.lower_bound_:.1f}")  # (3)
Xu = np.vstack([rng.multivariate_normal(m, [[0.6, 0.2], [0.2, 0.5]], 100) for m in pusat])
print(f"log-lik. uji per titik: EM {em.score(Xu):.3f}, VB {vb.score(Xu):.3f}")

## Listing 14.3: Model topik LDA untuk berita daerah sintetis

Berkas: `kode/bab14/lda_berita.py`

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from data_berita import buat_korpus, TOPIK

dok, theta, nama = buat_korpus()
latih, uji = dok[:1000], dok[1000:]
cv = CountVectorizer(token_pattern=r"[a-z]+")
Xl, Xu = cv.fit_transform(latih), cv.transform(uji)       # bag of words  # (1)
kata = cv.get_feature_names_out()

lda = LatentDirichletAllocation(6, doc_topic_prior=0.1, topic_word_prior=0.01,
                                learning_method="batch", random_state=0).fit(Xl)  # (2)
label = []
for k, b in enumerate(lda.components_):                   # kata teratas  # (3)
    atas = kata[np.argsort(b)[::-1][:6]]
    label.append(max(TOPIK, key=lambda t: len(set(atas) & set(TOPIK[t].split()))))
    print(f"topik {k} ({label[-1]:7s}): " + " ".join(atas))
d = 1000 + np.argmax(theta[1000:].max(1) < 0.7)         # dokumen uji campuran  # (4)
th = lda.transform(Xu[d - 1000])[0]
est = {label[k]: p for k, p in enumerate(th)}
print(f"dokumen #{d}  " + " ".join(f"{t:>7s}" for t in nama))
print("  taksiran    " + " ".join(f"{est.get(t, 0):7.2f}" for t in nama))
print("  sebenarnya  " + " ".join(f"{p:7.2f}" for p in theta[d]))
for K in [3, 6, 12]:                                      # perplexity data uji
    m = LatentDirichletAllocation(K, random_state=0).fit(Xl)
    print(f"K = {K:2d}: perplexity uji {m.perplexity(Xu):.1f}")